# Authors
Kseniia Ostudina (kseniia.ostudina@universite-paris-saclay.fr)

Marina Hornero Merino (marina.hornero-merino@universite-paris-saclay.fr)

Julie Dornat (julie.dornat@universite-paris-saclay.fr)

In [ ]:
!pip install gensim
!pip install fasttext

# Imports

In [ ]:
import urllib.request
import zipfile
import fasttext
import os
import time

import numpy as np
import pandas as pd
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.word2vec import LineSentence
from scipy.spatial import distance
from collections import Counter


In [ ]:
# Extract data

# Define the URL and the target filename
url = 'https://perso.limsi.fr/neveol/TP_ISD2020.zip'
zip_filename = 'TP_ISD2020.zip'

# Download the file from the link
urllib.request.urlretrieve(url, zip_filename)

# Extract the contents into a folder named 'extracted_data'
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('extracted_data')

print('Download and extraction complete!')

Download and extraction complete!


In [ ]:
# Create output directory
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_DIR = "extracted_data"
PRESS_DIR = f"{DATA_DIR}/QUAERO_FrenchPress"
MED_DIR = f"{DATA_DIR}/QUAERO_FrenchMed"
WORDS = ["patient", "traitement", "maladie", "solution", "jaune"]


# FastText Embedding (CBow)


In [ ]:
class FastText_Embedding():
  def __init__(self,
               emb_model="cbow",
               model_path=f"{OUTPUT_DIR}/fattest_embeddings.bin",
               load_model=False):
    self.emb_model = emb_model
    self.model_path = model_path
    if load_model and os.path.exists(self.model_path):
      self.model = fasttext.load_model(model_path)

  def fit(self, train_set: str = "QUAERO_FrenchMed_traindev.ospl"):
    self.model = fasttext.train_unsupervised(
        train_set,
        model=self.emb_model,
        dim=100,
        minCount=1,
    )
    self.model.save_model(self.model_path)
    return self

  def embed(self, word:str):
    # Predict the embedding of the word
    return self.model[word]

  def similar_words(self, word:str):
    return self.model.get_nearest_neighbors(word)



In [ ]:
med_train = f"{MED_DIR}/QUAERO_FrenchMed_traindev.ospl"
press_train = f"{PRESS_DIR}/QUAERO_FrenchPress_traindev.ospl"

Med_Emb = FastText_Embedding(model_path=f"{OUTPUT_DIR}/fasttext_med.bin").fit(
    train_set=med_train)
Press_Emb = FastText_Embedding(model_path=f"{OUTPUT_DIR}/fasttext_press.bin").fit(
    train_set=press_train)


In [ ]:
print("Med vocabulary size:", len(Med_Emb.model.get_words()))
print("Press vocabulary size:", len(Press_Emb.model.get_words()))

Med vocabulary size: 9105
Press vocabulary size: 39655


In [ ]:
print(len(Med_Emb.embed("rideaux"))) # should return a 100-vector
print(len(Press_Emb.embed("rideaux"))) # should return a 100-vector

100
100


# Embeddings with WORD2VEC

In [ ]:
for p in [med_train, press_train]:
    print(("OK   " if os.path.exists(p) else "MISS "), p)
assert os.path.exists(med_train) and os.path.exists(press_train), "Input files not found - check DATA"

CORPORA = {"med": med_train, "press": press_train}

OUT_DIR = os.path.join(DATA_DIR, "word2vec_output")
EMB_DIR = os.path.join(OUT_DIR, "embeddings")
RESULTS_DIR = os.path.join(OUT_DIR, "results")
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Outputs will be saved to:", OUT_DIR)

OK    extracted_data/QUAERO_FrenchMed/QUAERO_FrenchMed_traindev.ospl
OK    extracted_data/QUAERO_FrenchPress/QUAERO_FrenchPress_traindev.ospl
Outputs will be saved to: extracted_data/word2vec_output


In [ ]:
# Show the first 3 raw lines of each corpus
for name, path in CORPORA.items():
    print(f"{name}: first 3 lines")
    with open(path, encoding="utf-8") as f:
        for _, line in zip(range(3), f):
            print(repr(line[:200]))
    print()

med: first 3 lines
'EMEA / H / C / 551 \n'
'PRIALT \n'
'Qu ’ est ce que Prialt ? \n'

press: first 3 lines
'Patricia Martin , que voici , que voilà ! oh , bonjour Nicolas Stoufflet . \n'
'France Inter , 7 heures . \n'
'le journal , Simon Tivolle . \n'



In [ ]:
# Corpus statistics: sentences, tokens, vocabulary size
counters, stats = {}, []
for name, path in CORPORA.items():
    cnt, n_sent = Counter(), 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            toks = line.split()
            if toks:
                n_sent += 1
                cnt.update(toks)
    counters[name] = cnt
    stats.append({"corpus": name, "sentences": n_sent, "tokens": sum(cnt.values()),
                  "vocabulary": len(cnt),
                  "words only seen once": sum(1 for c in cnt.values() if c == 1)})

corpus_stats = pd.DataFrame(stats).set_index("corpus")
corpus_stats.to_csv(os.path.join(RESULTS_DIR, "corpus_stats.csv"))
corpus_stats

,sentences,tokens,vocabulary,words only seen once
corpus,,,,
med,3021,51896,9104,5594
press,38548,1251561,39654,15402


In [ ]:
# Frequency of the 5 target words in each corpus
freq = pd.DataFrame({name: [counters[name][w] for w in WORDS] for name in CORPORA}, index=WORDS)
freq.index.name = "word"
freq.to_csv(os.path.join(RESULTS_DIR, "word_frequencies.csv"))
freq

,med,press
word,,
patient,33,11
traitement,251,53
maladie,65,114
solution,67,100
jaune,9,23


The Med corpus is small, and the Press corpus is approximately 24 times bigger. However, medical words appear more often in Med (traitement: 251 vs 53). Some words are very rare (jaune: 9 in Med), so their results may not be reliable.

# Training the 4 word2vec models

Key parameter sg (skipgram): with **sg=0** model becomes **CBOW**, that is, predicting the word from its context; with **sg=1** the model becomes **Skip-gram** - predict the context from the word

In [ ]:
PARAMS = dict(vector_size=100, min_count=1, window=5, epochs=5, workers=4, seed=42)
ALGOS = {"cbow": 0, "skipgram": 1}

models, train_log = {}, []
# first time it trains on (med, cbow), then (med, skipgram), then (press, cbow)
# and finally (press, skipgram)
for corpus, path in CORPORA.items():
    for algo, sg in ALGOS.items():
        key = f"w2v_{algo}_{corpus}"
        print(f"Training {key} ...")
        t0 = time.time()
        model = Word2Vec(sentences=LineSentence(path), sg=sg, **PARAMS)
        secs = round(time.time() - t0, 1)

        model.wv.save_word2vec_format(os.path.join(EMB_DIR, f"{key}.vec"), binary=False)

        models[key] = model
        train_log.append({"model": key, "corpus": corpus, "algo": algo, "sg": sg,
                          "vocab_size": len(model.wv), "corpus_tokens": model.corpus_total_words,
                          "train_seconds": secs})
        print(f"  done: vocab={len(model.wv)}, {secs}s")

train_df = pd.DataFrame(train_log).set_index("model")
# train_df.to_csv(os.path.join(RESULTS_DIR, "training_log.csv"))

train_df

Training w2v_cbow_med ...
  done: vocab=9104, 0.4s
Training w2v_skipgram_med ...
  done: vocab=9104, 0.7s
Training w2v_cbow_press ...
  done: vocab=39654, 6.5s
Training w2v_skipgram_press ...
  done: vocab=39654, 17.4s


,corpus,algo,sg,vocab_size,corpus_tokens,train_seconds
model,,,,,,
w2v_cbow_med,med,cbow,0,9104,51896,0.4
w2v_skipgram_med,med,skipgram,1,9104,51896,0.7
w2v_cbow_press,press,cbow,0,39654,1251561,6.5
w2v_skipgram_press,press,skipgram,1,39654,1251561,17.4


In [ ]:
# List the saved embedding files
for fn in sorted(os.listdir(EMB_DIR)):
    print(f"{fn:35s} {os.path.getsize(os.path.join(EMB_DIR, fn)) / 1e6:8.1f} MB")

w2v_cbow_med.vec                        11.7 MB
w2v_cbow_press.vec                      49.6 MB
w2v_skipgram_med.vec                    11.3 MB
w2v_skipgram_press.vec                  47.6 MB


# Evaluation: Semantic similarity

## Method: scipy distance

We chose to do it with scipy because fasttext was developed using the *fasttext* library instead of *gensim*

In [ ]:
models_w2v = models
models_fasttext = {}
models_fasttext["fasttext_Med_Emb"] = Med_Emb.model
models_fasttext["fasttext_Press_Emb"] = Press_Emb.model
print(models_w2v)
print(models_fasttext)

{'w2v_cbow_med': <gensim.models.word2vec.Word2Vec object at 0x7851ed1d2e90>, 'w2v_skipgram_med': <gensim.models.word2vec.Word2Vec object at 0x7851b35479d0>, 'w2v_cbow_press': <gensim.models.word2vec.Word2Vec object at 0x7851b3545810>, 'w2v_skipgram_press': <gensim.models.word2vec.Word2Vec object at 0x7851b3544690>}
{'fasttext_Med_Emb': <fasttext.FastText._FastText object at 0x7851ed23f230>, 'fasttext_Press_Emb': <fasttext.FastText._FastText object at 0x78520bead490>}


In [ ]:
def closest_words_per_model(word, model, model_name):
    results = []

    if word in model.wv:
        vector = model.wv[word].reshape(1, -1)

        cosine_dist = distance.cdist(
            vector,
            model.wv.vectors,
            metric='cosine'
        )[0]

        sorted_indices = np.argsort(cosine_dist)[1:11]

        for rank, i in enumerate(sorted_indices, 1):
            closest_word = model.wv.index_to_key[i]

            results.append({
                'model_name': model_name,
                'target_word': word,
                'rank': rank,
                'closest_word': closest_word,
                'distance': cosine_dist[i]
            })

    else:
        results.append({
            'model_name': model_name,
            'target_word': word,
            'rank': np.nan,
            'closest_word': f"Word '{word}' not found",
            'distance': np.nan
        })

    return results


def closest_words_per_model_fasttext(
    word,
    model,
    model_name,
    all_words,
    all_vectors
):
    results = []

    if word in all_words:

        word_vector = model.get_word_vector(word).reshape(1, -1)

        cosine_dist = distance.cdist(
            word_vector,
            all_vectors,
            metric='cosine'
        )[0]

        sorted_indices = np.argsort(cosine_dist)[1:11]

        for rank, i in enumerate(sorted_indices, 1):

            results.append({
                'model_name': model_name,
                'target_word': word,
                'rank': rank,
                'closest_word': all_words[i],
                'distance': cosine_dist[i]
            })

    else:
        results.append({
            'model_name': model_name,
            'target_word': word,
            'rank': np.nan,
            'closest_word': f"Word '{word}' not found",
            'distance': np.nan
        })

    return results


all_results = []


# Word2Vec models

for model_name, model in models_w2v.items():

    for word in WORDS:

        all_results.extend(
            closest_words_per_model(
                word,
                model,
                model_name
            )
        )


# FastText models

for model_name, model in models_fasttext.items():

    all_words = model.get_words()

    all_vectors = np.array([
        model.get_word_vector(word)
        for word in all_words
    ])

    for word in WORDS:

        all_results.extend(
            closest_words_per_model_fasttext(
                word,
                model,
                model_name,
                all_words,
                all_vectors
            )
        )


results_df = pd.DataFrame(all_results)


### Corpus-based evaluation
Here we present the 10 closest words to the given words for each model, comparing the models based on the corpus it was trained on. On the left we see models trained on the medical corpus and on the right the ones trained on the press corpus.

In [ ]:
from IPython.display import display_html

models_med = ['w2v_cbow_med', 'w2v_skipgram_med', 'fasttext_Med_Emb']
models_press = ['w2v_cbow_press', 'w2v_skipgram_press', 'fasttext_Press_Emb']

# Prepare the data for pivoting
results_df['closest_word_distance'] = (
    results_df['closest_word']
    + ' ('
    + results_df['distance'].round(4).astype(str)
    + ')'
)

for word in WORDS:
    print(f"### Closest words for '{word}'\n")

    word_df = results_df[results_df['target_word'] == word]

    # Create pivot tables
    pivot_med = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    pivot_press = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    # add the med-trained models next to the press-trained ones for easy comparison
    med_columns = [m for m in models_med if m in pivot_med.columns]
    press_columns = [m for m in models_press if m in pivot_press.columns]

    pivot_med = pivot_med[med_columns].fillna('N/A')
    pivot_press = pivot_press[press_columns].fillna('N/A')

    # Add corpus labels above the tables
    html = f"""
    <div style="display: flex; gap: 30px; align-items: flex-start;">
        <div>
            <h4 style="text-align: center;">Medical Corpus</h4>
            {pivot_med.to_html()}
        </div>
        <div>
            <h4 style="text-align: center;">Press Corpus</h4>
            {pivot_press.to_html()}
        </div>
    </div>
    """

    display_html(html, raw=True)
    print("\n")

### Closest words for 'patient'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,sont (0.0008),qui (0.0035),patiente (0.0)
2,les (0.0008),symptômes (0.0036),patients (0.0)
3,le (0.0008),réactions (0.0046),appartient (0.0)
4,et (0.0008),grossesse (0.0054),entrainement (0.0)
5,que (0.0008),particulier (0.0054),département (0.0)
6,en (0.0009),ce (0.0055),conscient (0.0)
7,du (0.0009),ayant (0.0057),isolement (0.0)
8,peut (0.0009),Tysabri (0.0058),Traitement (0.0)
9,par (0.0009),instauration (0.0064),Département (0.0)




### Closest words for 'traitement'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,dans (0.0003),TYSABRI (0.024),avancement (0.0)
2,Le (0.0004),médicaments (0.0251),département (0.0)
3,TYSABRI (0.0004),association (0.0256),Département (0.0)
4,que (0.0004),par (0.0292),traitements (0.0)
5,avec (0.0004),le (0.0296),Traitement (0.0)
6,les (0.0004),autres (0.0309),ajustement (0.0)
7,des (0.0004),utilisation (0.0313),principalement (0.0)
8,par (0.0004),risque (0.0325),Ajustements (0.0)
9,le (0.0004),maladie (0.0351),chimiquement (0.0)




### Closest words for 'maladie'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,la (0.0004),infection (0.005),préopératoires (0.0)
2,du (0.0004),administration (0.0069),musculaires (0.0)
3,de (0.0005),après (0.0071),péricentriolaire (0.0)
4,un (0.0005),adulte (0.0084),conduire (0.0)
5,. (0.0005),utilisation (0.0093),territoires (0.0)
6,le (0.0005),VIH (0.01),circulatoires (0.0)
7,à (0.0005),association (0.0105),postopératoires (0.0)
8,en (0.0005),homme (0.0105),secondaires (0.0)
9,des (0.0005),autres (0.0111),ventriculaires (0.0)




### Closest words for 'solution'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,( (0.0008),flacon (0.0204),solutions (0.0)
2,) (0.0008),300 (0.0241),Dissolution (0.0)
3,pour (0.0008),fois (0.0265),evolution (0.0)
4,ou (0.0008),150 (0.0268),Solution (0.0)
5,perfusion (0.0008),contient (0.027),exception (0.0)
6,2 (0.0009),perfusion (0.0271),contraception (0.0)
7,", (0.0009)",jour (0.0274),exécution (0.0)
8,flacon (0.0009),100 (0.0278),congélation (0.0)
9,à (0.0009),mg (0.0315),Evolution (0.0)




### Closest words for 'jaune'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,) (0.0031),64 (0.0019),identique (0.0)
2,. (0.0032),eau (0.002),orientables (0.0)
3,: (0.0032),heures (0.002),neuropathique (0.0)
4,3 (0.0033),jours (0.0021),bicompartimental (0.0)
5,• (0.0033),21 (0.0025),diagnostique (0.0)
6,( (0.0033),chlorure (0.0027),typiquement (0.0)
7,", (0.0033)",environ (0.0027),intention (0.0)
8,lamivudine (0.0033),421 (0.0027),Franceville (0.0)
9,à (0.0034),24 (0.0027),historique (0.0)


Interpretation: we can see that the model has created very similar embeddings for words in the medical domain (for the medical corpus) and words in the press domain. For example, for 'patient', we can see that the closest word in the medical domain, the skipgram and fasttext give words like 'symptômes', 'recommandé', 'Traitement' and so on. Meanwhile, the same models trained on a press corpus give words that could be more likely to see in news articles, like 'lecteur', or 'idéologie'. There's a notable difference with CBOW, which returns shorts or stopwords as closest words for any given word. But, in general, the closest words returned per corpus are words that are expected in each specific domain.

### Model-based evaluation
Same results but aggregated by model type.

In [ ]:
models_cbow = ['w2v_cbow_med', 'w2v_cbow_press']
models_skipgram = ['w2v_skipgram_med', 'w2v_skipgram_press']
models_fasttext = ['fasttext_Med_Emb', 'fasttext_Press_Emb']

# Prepare the data for pivoting
results_df['closest_word_distance'] = (
    results_df['closest_word']
    + ' ('
    + results_df['distance'].round(4).astype(str)
    + ')'
)

for word in WORDS:
    print(f"### Closest words for '{word}'\n")

    word_df = results_df[results_df['target_word'] == word]

    # Create pivot tables
    pivot_cbow = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    pivot_skipgram = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    pivot_fasttext = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    # add the med-trained models next to the press-trained ones for easy comparison
    cbow_columns = [m for m in models_cbow if m in pivot_cbow.columns]
    skipgram_columns = [m for m in models_skipgram if m in pivot_skipgram.columns]
    fasttext_columns = [m for m in models_fasttext if m in pivot_fasttext.columns]

    pivot_cbow = pivot_cbow[cbow_columns].fillna('N/A')
    pivot_skipgram = pivot_skipgram[skipgram_columns].fillna('N/A')
    pivot_fasttext = pivot_fasttext[fasttext_columns].fillna('N/A')

    # Add corpus labels above the tables
    html = f"""
    <div style="display: flex; gap: 30px; align-items: flex-start;">
        <div>
            <h4 style="text-align: center;">Models using Continuous Bag of Words (CBOW)</h4>
            {pivot_cbow.to_html()}
        </div>
        <div>
            <h4 style="text-align: center;">Models using skipgram</h4>
            {pivot_skipgram.to_html()}
        </div>
        <div>
            <h4 style="text-align: center;">Press Corpus</h4>
            {pivot_fasttext.to_html()}
        </div>
    </div>
    """

    display_html(html, raw=True)
    print("\n")

### Closest words for 'patient'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,sont (0.0008),style (0.1293)
2,les (0.0008),pont (0.1409)
3,le (0.0008),scénario (0.1418)
4,et (0.0008),chemin (0.1421)
5,que (0.0008),parcours (0.1435)
6,en (0.0009),défi (0.1445)
7,du (0.0009),cimetière (0.1526)
8,peut (0.0009),personnage (0.1547)
9,par (0.0009),instrument (0.1552)




### Closest words for 'traitement'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,dans (0.0003),financement (0.0529)
2,Le (0.0004),calendrier (0.0606)
3,TYSABRI (0.0004),viol (0.0766)
4,que (0.0004),luxe (0.084)
5,avec (0.0004),consensus (0.0841)
6,les (0.0004),pacte (0.0855)
7,des (0.0004),permanent (0.0883)
8,par (0.0004),matériel (0.0899)
9,le (0.0004),mépris (0.0916)




### Closest words for 'maladie'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,la (0.0004),puissance (0.0841)
2,du (0.0004),lumière (0.1025)
3,de (0.0005),motion (0.107)
4,un (0.0005),montée (0.1096)
5,. (0.0005),commande (0.1105)
6,le (0.0005),protestation (0.1126)
7,à (0.0005),mondialisation (0.1159)
8,en (0.0005),remise (0.1162)
9,des (0.0005),construction (0.1174)




### Closest words for 'solution'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,( (0.0008),règle (0.08)
2,) (0.0008),catastrophe (0.0823)
3,pour (0.0008),seule (0.0839)
4,ou (0.0008),démarche (0.0866)
5,perfusion (0.0008),forme (0.1028)
6,2 (0.0009),tradition (0.1046)
7,", (0.0009)",dimension (0.1069)
8,flacon (0.0009),source (0.1085)
9,à (0.0009),conclusion (0.1116)




### Closest words for 'jaune'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,) (0.0031),maillot (0.0461)
2,. (0.0032),Léa (0.0525)
3,: (0.0032),B (0.053)
4,3 (0.0033),populiste (0.0572)
5,• (0.0033),Amir (0.0604)
6,( (0.0033),mea (0.0609)
7,", (0.0033)",Edouard (0.0611)
8,lamivudine (0.0033),libéral (0.0626)
9,à (0.0034),voile (0.064)


Interpretation: here it is easier to compare the behavior of each model. For example, we can detect that fasttext tends to associate words that have similar ending to the given word, since it represents words using information learned from subwords. And cbow returns words or characters that tend to happen nearby, like skipgram, although they learn contextual information differently.

We notice that fasttext tends to produce smaller distances for each word (distance of 0 in some cases). Models trained on CBOW provide also close distances but less than fasttext. And finally skipgram models' embeddings show larger distances to their closest neighbors.

Another common pattern can be observed between corpora. The embeddings of all models trained on the medical domain report closer distances than the same model trained on the press domain. This can be due to the bigger size of the press corpus (it contains over 4 times the amount of words present in the medical corpus, and over 3 times the amout of unique words). This may result in less concentrated embedding space and therefore larger distances between target words and nearest neighbors.  